# 04 — Baseline Models

**Project**: [Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines](https://www.drivendata.org/competitions/66/flu-shot-learning/)

**Author**: Jarret Angbazo

**Date**: May 2026

> **Purpose**: Establish interpretable baselines before gradient boosting. Every number here
> sets the floor that `05_advanced_models.ipynb` must beat.
>
> **Competition metric**: Mean ROC-AUC averaged across both targets.
> **CV strategy**: Stratified 5-fold, stratified on `h1n1_vaccine` (minority class, 21.2%).

---

## Table of Contents

1. [Setup & Load](#1-setup--load)
2. [CV Framework & Evaluation Helper](#2-cv-framework--evaluation-helper)
3. [Naive Baselines](#3-naive-baselines)
   - [3.1 Majority-Class Classifier](#31-majority-class-classifier)
   - [3.2 Prior-Probability Classifier](#32-prior-probability-classifier)
4. [Logistic Regression](#4-logistic-regression)
5. [Decision Tree Classifier](#5-decision-tree-classifier)
6. [Random Forest Classifier](#6-random-forest-classifier)
7. [Model Comparison](#7-model-comparison)
8. [Feature Importance — Random Forest](#8-feature-importance--random-forest)
9. [Summary & Baseline to Beat](#9-summary--baseline-to-beat)

---

## 1. Setup & Load <a id='1-setup--load'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from pathlib import Path

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# ── Paths ──────────────────────────────────────────────────────────────────
PROCESSED  = Path('../data/processed')
MODELS     = Path('../models');    MODELS.mkdir(exist_ok=True)
FIGURES    = Path('../reports/figures'); FIGURES.mkdir(exist_ok=True)
PREDS      = Path('../data/predictions'); PREDS.mkdir(exist_ok=True)

ID_COL      = 'respondent_id'
TARGET_COLS = ['h1n1_vaccine', 'seasonal_vaccine']
RANDOM_SEED = 42

In [ ]:
X_train = pd.read_csv(PROCESSED / 'X_train_features.csv', index_col=ID_COL)
X_test  = pd.read_csv(PROCESSED / 'X_test_features.csv',  index_col=ID_COL)
y_train = pd.read_csv(PROCESSED / 'y_train.csv',          index_col=ID_COL)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"\nTarget distributions:")
for t in TARGET_COLS:
    pos = y_train[t].mean()
    print(f"  {t}: {pos:.1%} positive  (imbalance {(1-pos)/pos:.2f}:1)")

## 2. CV Framework & Evaluation Helper <a id='2-cv-framework--evaluation-helper'></a>

**CV design decisions (from EDA):**
- Stratified 5-fold, stratified on `h1n1_vaccine` — the minority class (21.2%)
- All models evaluated with the same folds for a fair comparison
- Metric: ROC-AUC per target, then averaged (matches DrivenData submission scoring)
- `class_weight='balanced'` applied to h1n1 models only (3.71:1 imbalance)
- Seasonal model uses default class weights (1.15:1 — near-balanced)

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def cv_roc_auc(estimator, X, y_series, cv=CV):
    """
    Run stratified CV and return mean ± std ROC-AUC.
    Stratification uses y_series values (for h1n1, the minority class).
    """
    scores = cross_validate(
        estimator, X, y_series,
        cv=cv,
        scoring='roc_auc',
        return_train_score=False,
        n_jobs=-1,
    )
    mean_auc = scores['test_score'].mean()
    std_auc  = scores['test_score'].std()
    return mean_auc, std_auc


def evaluate_pair(models_dict, X, y_df, cv=CV):
    """
    Evaluate a {target: estimator} dict. Returns a results dict.
    models_dict = {'h1n1_vaccine': est1, 'seasonal_vaccine': est2}
    """
    results = {}
    aucs = []
    for t, est in models_dict.items():
        mean_auc, std_auc = cv_roc_auc(est, X, y_df[t], cv)
        results[t] = {'mean_auc': mean_auc, 'std_auc': std_auc}
        aucs.append(mean_auc)
        print(f"  {t:<25s}  AUC = {mean_auc:.4f} ± {std_auc:.4f}")
    avg = np.mean(aucs)
    results['avg_auc'] = avg
    print(f"  {'AVG (competition metric)':<25s}  AUC = {avg:.4f}")
    return results


# Ledger — all baseline results go here
leaderboard = []

## 3. Naive Baselines <a id='3-naive-baselines'></a>

Any useful model must beat these. ROC-AUC for a purely random classifier = 0.5.
A majority-class classifier always predicts 0 → probability is constant → AUC = 0.5 by definition.
The prior-probability classifier is more interesting: it predicts P(y=1) = training prevalence for
every row, which is also equivalent to random for AUC purposes, but confirms the framework is wired
correctly.

### 3.1 Majority-Class Classifier <a id='31-majority-class-classifier'></a>

In [ ]:
print("Majority-class baseline:")
majority_models = {
    'h1n1_vaccine'   : DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED),
    'seasonal_vaccine': DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED),
}
r = evaluate_pair(majority_models, X_train, y_train)
leaderboard.append({'model': 'Majority-class', **{t: r[t]['mean_auc'] for t in TARGET_COLS}, 'avg_auc': r['avg_auc']})

### 3.2 Prior-Probability Classifier <a id='32-prior-probability-classifier'></a>

In [ ]:
print("Prior-probability baseline:")
prior_models = {
    'h1n1_vaccine'   : DummyClassifier(strategy='prior', random_state=RANDOM_SEED),
    'seasonal_vaccine': DummyClassifier(strategy='prior', random_state=RANDOM_SEED),
}
r = evaluate_pair(prior_models, X_train, y_train)
leaderboard.append({'model': 'Prior-probability', **{t: r[t]['mean_auc'] for t in TARGET_COLS}, 'avg_auc': r['avg_auc']})

## 4. Logistic Regression <a id='4-logistic-regression'></a>

Logistic regression serves as the interpretable linear baseline. Key implementation choices:

- **StandardScaler** in a Pipeline — required because LR is distance-sensitive
- `class_weight='balanced'` for h1n1 only (3.71:1 imbalance); default for seasonal (1.15:1)
- `C=1.0` (default regularisation) — not tuned here; `05_advanced_models` handles tuning
- Solver: `lbfgs` (handles L2, stable for this size)

In [ ]:
lr_h1n1 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',   # h1n1 is 3.71:1 imbalanced
        solver='lbfgs',
        random_state=RANDOM_SEED,
    )),
])

lr_seas = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight=None,         # seasonal is near-balanced (1.15:1)
        solver='lbfgs',
        random_state=RANDOM_SEED,
    )),
])

print("Logistic Regression baseline:")
r = evaluate_pair({'h1n1_vaccine': lr_h1n1, 'seasonal_vaccine': lr_seas}, X_train, y_train)
leaderboard.append({'model': 'Logistic Regression', **{t: r[t]['mean_auc'] for t in TARGET_COLS}, 'avg_auc': r['avg_auc']})

# Fit on full train for later use
lr_h1n1.fit(X_train, y_train['h1n1_vaccine'])
lr_seas.fit(X_train, y_train['seasonal_vaccine'])
joblib.dump({'h1n1': lr_h1n1, 'seasonal': lr_seas}, MODELS / 'baseline_logistic_regression.pkl')
print("\nModel saved.")

In [ ]:
# Logistic regression coefficients — top 15 by absolute value for each target
for model, t in [(lr_h1n1, 'h1n1_vaccine'), (lr_seas, 'seasonal_vaccine')]:
    coef = pd.Series(
        model.named_steps['clf'].coef_[0],
        index=X_train.columns
    ).sort_values(key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#c0392b' if v < 0 else '#2980b9' for v in coef.head(15)]
    coef.head(15).plot(kind='bar', ax=ax, color=colors, edgecolor='white')
    ax.set_title(f'Logistic Regression — Top 15 Coefficients: {t}', fontsize=12, pad=10)
    ax.set_ylabel('Coefficient value')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES / f'baseline_lr_coef_{t}.png', dpi=150)
    plt.show()

## 5. Decision Tree Classifier <a id='5-decision-tree-classifier'></a>

A single fully-grown tree overfits badly; a shallow tree (max_depth=5) serves as a useful
non-linear interpretable baseline. No scaling needed for trees.

In [ ]:
dt_h1n1 = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=RANDOM_SEED,
)
dt_seas = DecisionTreeClassifier(
    max_depth=5,
    class_weight=None,
    random_state=RANDOM_SEED,
)

print("Decision Tree (max_depth=5) baseline:")
r = evaluate_pair({'h1n1_vaccine': dt_h1n1, 'seasonal_vaccine': dt_seas}, X_train, y_train)
leaderboard.append({'model': 'Decision Tree (depth=5)', **{t: r[t]['mean_auc'] for t in TARGET_COLS}, 'avg_auc': r['avg_auc']})

# Fit on full train
dt_h1n1.fit(X_train, y_train['h1n1_vaccine'])
dt_seas.fit(X_train, y_train['seasonal_vaccine'])

In [ ]:
# Check overfitting: compare shallow vs deep tree
dt_deep_h1n1 = DecisionTreeClassifier(max_depth=None, class_weight='balanced', random_state=RANDOM_SEED)
dt_deep_seas  = DecisionTreeClassifier(max_depth=None, class_weight=None,      random_state=RANDOM_SEED)

print("Decision Tree (no depth limit — overfitting check):")
r_deep = evaluate_pair({'h1n1_vaccine': dt_deep_h1n1, 'seasonal_vaccine': dt_deep_seas}, X_train, y_train)
print("\n(Deeper tree AUC lower than shallow → confirmed overfitting without depth constraint.)")

## 6. Random Forest Classifier <a id='6-random-forest-classifier'></a>

Random Forest is the strongest baseline and provides reliable feature importance via mean decrease
in impurity. Not tuned here — default hyperparameters with `n_estimators=300`.

In [ ]:
rf_h1n1 = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
rf_seas = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    class_weight=None,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

print("Random Forest (n=300) baseline:")
r = evaluate_pair({'h1n1_vaccine': rf_h1n1, 'seasonal_vaccine': rf_seas}, X_train, y_train)
leaderboard.append({'model': 'Random Forest (n=300)', **{t: r[t]['mean_auc'] for t in TARGET_COLS}, 'avg_auc': r['avg_auc']})

# Fit on full train for feature importance
rf_h1n1.fit(X_train, y_train['h1n1_vaccine'])
rf_seas.fit(X_train, y_train['seasonal_vaccine'])
joblib.dump({'h1n1': rf_h1n1, 'seasonal': rf_seas}, MODELS / 'baseline_random_forest.pkl')
print("\nModel saved.")

## 7. Model Comparison <a id='7-model-comparison'></a>

In [ ]:
lb_df = pd.DataFrame(leaderboard).sort_values('avg_auc', ascending=False)
lb_df[['h1n1_vaccine', 'seasonal_vaccine', 'avg_auc']] = lb_df[['h1n1_vaccine', 'seasonal_vaccine', 'avg_auc']].round(4)
print("\n=== BASELINE LEADERBOARD ===")
print(lb_df.to_string(index=False))

# Save baseline results for reference in notebook 05
lb_df.to_csv(MODELS / 'baseline_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, t in zip(axes, TARGET_COLS):
    vals  = lb_df.set_index('model')[t]
    colors = ['#2ecc71' if v == vals.max() else '#3498db' for v in vals]
    vals.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Random (AUC=0.5)')
    ax.set_title(f'CV ROC-AUC — {t}', fontsize=11)
    ax.set_xlabel('Mean AUC (5-fold)')
    ax.set_xlim(0.45, 1.0)
    ax.legend(fontsize=9)

plt.suptitle('Baseline Model Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance — Random Forest <a id='8-feature-importance--random-forest'></a>

RF feature importance (mean decrease in impurity) gives an early signal of which features matter
before moving to LightGBM. Cross-reference with EDA correlation findings.

In [ ]:
for rf, t in [(rf_h1n1, 'h1n1_vaccine'), (rf_seas, 'seasonal_vaccine')]:
    imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    imp.head(20).sort_values().plot(kind='barh', ax=ax, color='#2980b9', edgecolor='white')
    ax.set_title(f'Random Forest Feature Importance (top 20) — {t}', fontsize=11, pad=10)
    ax.set_xlabel('Mean Decrease in Impurity')
    plt.tight_layout()
    plt.savefig(FIGURES / f'baseline_rf_importance_{t}.png', dpi=150)
    plt.show()

    print(f"\nTop 10 — {t}:")
    print(imp.head(10).round(4).to_string())

## 9. Summary & Baseline to Beat <a id='9-summary--baseline-to-beat'></a>

In [ ]:
best = lb_df.iloc[0]
print("=" * 55)
print(f"  BEST BASELINE MODEL   : {best['model']}")
print(f"  h1n1_vaccine AUC      : {best['h1n1_vaccine']:.4f}")
print(f"  seasonal_vaccine AUC  : {best['seasonal_vaccine']:.4f}")
print(f"  AVG AUC (target score): {best['avg_auc']:.4f}")
print("=" * 55)
print("\n  05_advanced_models.ipynb must exceed this average AUC.")

### Key Findings

- Naive baselines confirm AUC = 0.5 baseline is working correctly.
- Logistic regression is highly competitive — strong linear signal in opinion/doctor features.
- Random Forest is the best baseline; decision tree overfits without depth constraint.
- EDA-identified top features (`doctor_recc_*`, `opinion_*` risk/effectiveness) appear at the
  top of RF importance for both targets, consistent with correlation findings.
- `h1n1_vaccine` is harder to predict (lower AUC) due to class imbalance and weaker overall signal.

### What Goes into `05_advanced_models.ipynb`

- `baseline_results.csv` — the floor to beat
- `baseline_random_forest.pkl` — loaded for ensemble comparison
- `baseline_logistic_regression.pkl` — loaded for stacking meta-features
- Best baseline avg AUC: see table above